# T-SQL Fundamentals Chapter 7 - Data Analysis Examples

In this notebook, we demonstrate various T-SQL data analysis techniques introduced in Chapter 7, including window functions for ranking and offset, aggregate window functions, the WINDOW clause, pivot/unpivot transformations, grouping sets, rollup, cube, and time series analysis with DATE_BUCKET.

## Query 1: Ranking salespeople by SalesYTD (ROW_NUMBER, RANK, DENSE_RANK)

This query ranks salespeople based on their year-to-date sales (SalesYTD). It uses the three ranking functions **ROW_NUMBER()**, **RANK()**, and **DENSE_RANK()** to assign rankings to each salesperson within their sales territory, ordered by SalesYTD descending. This allows comparison of how each function handles ties in SalesYTD.

In [ ]:
SELECT 
    TerritoryName,
    FirstName, LastName,
    SalesYTD,
    ROW_NUMBER() OVER(PARTITION BY TerritoryName ORDER BY SalesYTD DESC) AS RowNumRank,
    RANK()       OVER(PARTITION BY TerritoryName ORDER BY SalesYTD DESC) AS Rank,
    DENSE_RANK() OVER(PARTITION BY TerritoryName ORDER BY SalesYTD DESC) AS DenseRank
FROM Sales.vSalesPerson
WHERE TerritoryName IS NOT NULL AND SalesYTD <> 0
ORDER BY TerritoryName, SalesYTD DESC;

## Query 2: Year-over-year sales per territory (LAG offset function)

This query calculates the year-over-year sales for each sales territory. It uses a **LAG()** window function to retrieve the previous year's sales for each territory, then computes the difference. This shows the sales trend by comparing each year's total sales to the prior year in the same territory.

In [ ]:
WITH TerritoryYearSales AS (
    SELECT t.Name AS Territory,
           YEAR(h.OrderDate) AS OrderYear,
           SUM(h.TotalDue) AS TotalSales
    FROM Sales.SalesOrderHeader AS h
    JOIN Sales.SalesTerritory AS t
      ON h.TerritoryID = t.TerritoryID
    GROUP BY t.Name, YEAR(h.OrderDate)
)
SELECT 
    Territory,
    OrderYear,
    TotalSales,
    LAG(TotalSales)  OVER(PARTITION BY Territory ORDER BY OrderYear) AS PrevYearSales,
    TotalSales - LAG(TotalSales) OVER(PARTITION BY Territory ORDER BY OrderYear) AS YoYChange
FROM TerritoryYearSales
ORDER BY Territory, OrderYear;

## Query 3: Sales vs. average sales per territory (aggregate window function)

This query compares each salesperson's SalesYTD to the average SalesYTD in their territory. It uses an aggregate window function **AVG(SalesYTD) OVER(PARTITION BY TerritoryName)** to compute the average sales per territory, and then shows how much each salesperson is above or below that average. This highlights relative performance within each territory.

In [ ]:
SELECT 
    TerritoryName,
    FirstName, LastName,
    SalesYTD,
    AVG(SalesYTD) OVER(PARTITION BY TerritoryName) AS AvgTerritorySales,
    SalesYTD - AVG(SalesYTD) OVER(PARTITION BY TerritoryName) AS DifferenceFromAvg
FROM Sales.vSalesPerson
WHERE TerritoryName IS NOT NULL
ORDER BY TerritoryName;

## Query 4: Cumulative annual sales by product category (WINDOW clause)

This query demonstrates the use of the **WINDOW** clause to define a reusable window specification. It calculates total sales per product category per year, and then uses a window definition to compute a running total of sales across years for each category. The WINDOW clause avoids repeating the PARTITION BY and ORDER BY clauses for the cumulative sum calculation.

In [ ]:
WITH CategoryYearSales AS (
    SELECT pc.Name AS Category,
           YEAR(h.OrderDate) AS SalesYear,
           SUM(d.LineTotal) AS YearlySales
    FROM Sales.SalesOrderDetail AS d
    JOIN Sales.SalesOrderHeader AS h
      ON d.SalesOrderID = h.SalesOrderID
    JOIN Production.Product AS p
      ON d.ProductID = p.ProductID
    JOIN Production.ProductSubcategory AS s
      ON p.ProductSubcategoryID = s.ProductSubcategoryID
    JOIN Production.ProductCategory AS pc
      ON s.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name, YEAR(h.OrderDate)
)
SELECT 
    Category,
    SalesYear,
    YearlySales,
    SUM(YearlySales) OVER w AS CumulativeSales
FROM CategoryYearSales
WINDOW w AS (PARTITION BY Category ORDER BY SalesYear)
ORDER BY Category, SalesYear;

## Query 5: Pivot: Total sales by year for each product category

This query pivots annual sales data to transform rows into columns. It shows total sales for each product category (Accessories, Bikes, Clothing, Components) for each year. The **PIVOT** operator is used on the product category column to create separate columns for each category's sales, aggregated by year.

In [ ]:
SELECT 
    Year,
    ISNULL([Accessories], 0) AS AccessoriesSales,
    ISNULL([Bikes], 0)       AS BikesSales,
    ISNULL([Clothing], 0)    AS ClothingSales,
    ISNULL([Components], 0)  AS ComponentsSales
FROM (
    SELECT YEAR(h.OrderDate) AS Year,
           pc.Name AS Category,
           d.LineTotal AS Sales
    FROM Sales.SalesOrderDetail AS d
    JOIN Sales.SalesOrderHeader AS h
      ON d.SalesOrderID = h.SalesOrderID
    JOIN Production.Product AS p
      ON d.ProductID = p.ProductID
    JOIN Production.ProductSubcategory AS s
      ON p.ProductSubcategoryID = s.ProductSubcategoryID
    JOIN Production.ProductCategory AS pc
      ON s.ProductCategoryID = pc.ProductCategoryID
) AS SourceData
PIVOT (
    SUM(Sales) 
    FOR Category IN ([Accessories], [Bikes], [Clothing], [Components])
) AS PivotTable
ORDER BY Year;

## Query 6: Unpivot: Product costs and prices as attribute-value rows

This query uses **UNPIVOT** to transform columns into rows. It takes each product's StandardCost and ListPrice, and unpivots them into an **Attribute** column and **Value** column. Each product will have one row for its StandardCost and another for its ListPrice, which is useful for comparing cost vs. price per product.

In [ ]:
SELECT 
    ProductID,
    Name,
    Attribute,
    Value
FROM (
    SELECT ProductID, Name, StandardCost, ListPrice
    FROM Production.Product
    WHERE StandardCost IS NOT NULL AND ListPrice IS NOT NULL
) AS Prod
UNPIVOT
    (Value FOR Attribute IN (StandardCost, ListPrice))
    AS Unpvt
ORDER BY ProductID, Attribute;

## Query 7: Grouping Sets: Sales totals by year and territory

This query uses a **GROUPING SETS** clause to produce multiple group-by aggregations in one result. It calculates total sales (TotalDue) grouped by year, by territory, and an overall total. The output includes rows for each year (with 'All Territories'), each territory (with 'All Years'), and a grand total ('All Years' and 'All Territories'). This demonstrates custom subtotal rows in a single query.

In [ ]:
SELECT 
    CASE WHEN GROUPING(YEAR(h.OrderDate)) = 1 THEN 'All Years'
         ELSE CAST(YEAR(h.OrderDate) AS varchar(4)) END AS Year,
    CASE WHEN GROUPING(t.Name) = 1 THEN 'All Territories'
         ELSE t.Name END AS Territory,
    SUM(h.TotalDue) AS TotalSales
FROM Sales.SalesOrderHeader AS h
JOIN Sales.SalesTerritory AS t
  ON h.TerritoryID = t.TerritoryID
GROUP BY GROUPING SETS (
    (YEAR(h.OrderDate)),
    (t.Name),
    ()
)
ORDER BY 
    CASE WHEN GROUPING(YEAR(h.OrderDate)) = 1 THEN 1 ELSE 0 END,
    YEAR(h.OrderDate),
    CASE WHEN GROUPING(t.Name) = 1 THEN 1 ELSE 0 END,
    t.Name;

## Query 8: Rollup: Product count by Category and Subcategory

This query uses **ROLLUP** to get hierarchical totals. It groups the count of products by ProductCategory and ProductSubcategory. Using ROLLUP(Category, Subcategory) produces rows for each subcategory, each category (total of subcategories), and an overall total of products. The result includes product counts for each subcategory, category-level totals (with 'All Subcategories'), and a grand total ('All Categories' with 'All Subcategories').

In [ ]:
SELECT 
    CASE WHEN GROUPING(pc.Name) = 1 THEN 'All Categories' ELSE pc.Name END AS Category,
    CASE WHEN GROUPING(s.Name) = 1 THEN 'All Subcategories' ELSE s.Name END AS SubCategory,
    COUNT(p.ProductID) AS ProductCount
FROM Production.ProductCategory AS pc
JOIN Production.ProductSubcategory AS s
  ON pc.ProductCategoryID = s.ProductCategoryID
JOIN Production.Product AS p
  ON s.ProductSubcategoryID = p.ProductSubcategoryID
GROUP BY ROLLUP(pc.Name, s.Name)
ORDER BY 
    CASE WHEN GROUPING(pc.Name) = 1 THEN 1 ELSE 0 END, 
    pc.Name,
    CASE WHEN GROUPING(s.Name) = 1 THEN 1 ELSE 0 END,
    s.Name;

## Query 9: Cube: Sales totals by year and territory (all combinations)

This query uses **CUBE** to generate all combinations of aggregations for year and territory. It produces total sales (TotalDue) for each year-territory combination, as well as subtotals for each year (across all territories), each territory (across all years), and a grand total. The result includes detailed year-by-territory sales, plus summary rows (with 'All Years' or 'All Territories') covering all combinations.

In [ ]:
SELECT 
    CASE WHEN GROUPING(YEAR(h.OrderDate)) = 1 THEN 'All Years'
         ELSE CAST(YEAR(h.OrderDate) AS varchar(4)) END AS Year,
    CASE WHEN GROUPING(t.Name) = 1 THEN 'All Territories'
         ELSE t.Name END AS Territory,
    SUM(h.TotalDue) AS TotalSales
FROM Sales.SalesOrderHeader AS h
JOIN Sales.SalesTerritory AS t
  ON h.TerritoryID = t.TerritoryID
GROUP BY CUBE(YEAR(h.OrderDate), t.Name)
ORDER BY 
    CASE WHEN GROUPING(YEAR(h.OrderDate)) = 1 THEN 1 ELSE 0 END,
    YEAR(h.OrderDate),
    CASE WHEN GROUPING(t.Name) = 1 THEN 1 ELSE 0 END,
    t.Name;

## Query 10: Weekly sales summary (DATE_BUCKET time series)

This query performs a time-series aggregation of orders by week using the **DATE_BUCKET** function (new in SQL Server 2022). It groups the SalesOrderHeader data into weekly buckets based on OrderDate and calculates the number of orders and total sales for each week. This helps analyze trends on a weekly basis.

In [ ]:
SELECT 
    DATE_BUCKET(WEEK, 1, OrderDate) AS WeekStart,
    COUNT(*) AS TotalOrders,
    SUM(TotalDue) AS TotalSales
FROM Sales.SalesOrderHeader
GROUP BY DATE_BUCKET(WEEK, 1, OrderDate)
ORDER BY WeekStart;

_*Note: This notebook was prepared with the assistance of ChatGPT, an AI language model.*_